In [34]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset

checkpoint = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_dataset = load_dataset("nyu-mll/glue", "mrpc")

In [35]:
def tokenize(dataset, tokenizer):
    return tokenizer(
        dataset['sentence1'],
        dataset['sentence2'],
        truncation=True
    )

In [36]:
tokenized_dataset = raw_dataset.map(tokenize, batched=True, fn_kwargs={'tokenizer': tokenizer})
tokenized_dataset = tokenized_dataset.remove_columns(['sentence1', 'sentence2', 'idx']).rename_column('label', 'labels')
tokenized_dataset.set_format('torch')

In [37]:
collator = DataCollatorWithPadding(tokenizer)

In [38]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=tokenized_dataset['train'],
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

eval_loader = DataLoader(
    dataset=tokenized_dataset['validation'],
    batch_size=16,
    collate_fn=collator
)

In [39]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 883.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [40]:
batch = next(iter(train_loader))

In [41]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.7540, grad_fn=<NllLossBackward0>) torch.Size([16, 2])


In [42]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [43]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [44]:
# loss = outputs.loss
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()

In [45]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)
#learning rate decay

In [46]:
from tqdm import tqdm

progress_bar = tqdm(total=num_training_steps)

model.train()
for epoch in range(num_epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        y_pred = model(**batch)
        loss = y_pred.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

100%|██████████| 690/690 [01:55<00:00,  5.88it/s]

In [47]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()

for batch in eval_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.inference_mode():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch['labels'])

metric.compute()

{'accuracy': 0.8382352941176471, 'f1': 0.8865979381443299}

also you can replace device switching easily using accelerator that handles it for you

In [48]:
from accelerate import Accelerator

accelerator = Accelerator()
model2 = AutoModelForSequenceClassification.from_pretrained(checkpoint)
optimizer2 = AdamW(model2.parameters(), lr=5e-5)
train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_loader, eval_loader, model2, optimizer2
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 985.80it/s][A
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [49]:
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [54]:
progress_bar = tqdm(
    total=num_epochs * len(train_dl),
)

  6%|▌         | 41/690 [00:09<02:27,  4.40it/s]


In [55]:
model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

100%|██████████| 690/690 [02:13<00:00,  4.92it/s]

In [56]:
metric = evaluate.load("glue", "mrpc")
model.eval()

with torch.inference_mode():
    for batch in eval_dl:
        outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=-1)

        predictions, labels = accelerator.gather_for_metrics(
            (predictions, batch["labels"])
        )

        metric.add_batch(
            predictions=predictions.cpu(),
            references=labels.cpu(),
        )

print(metric.compute())

{'accuracy': 0.8431372549019608, 'f1': 0.8929765886287625}


In [57]:
import numpy as np
import evaluate

metric = evaluate.load("glue", "mrpc")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [61]:
%wandb login

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: WARNING Invalid choice
wandb: WARNING Invalid choice
wandb: WARNING Invalid choice
wandb: WARNING Invalid choice



KeyboardInterrupt: 

In [62]:
from transformers import Trainer, TrainingArguments
import wandb

wandb.init(project='transformer-fine-tuning', name='bert-mrpc-analysis')

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    logging_steps=10,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to='wandb'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\user\_netrc.
wandb: Currently logged in as: abdenalisher (abdenalisher-sdu-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,F1
50,0.406756,0.517961,0.850490,0.894281
100,0.164688,0.896571,0.813725,0.877814
150,0.178585,0.653724,0.838235,0.890728
200,0.148190,0.599566,0.845588,0.892675
250,0.078921,0.739558,0.830882,0.881647
300,0.004057,0.840796,0.843137,0.890411
350,0.038370,0.746449,0.838235,0.889262
400,0.036010,0.769775,0.860294,0.903879
450,0.008600,0.771617,0.838235,0.884211
500,0.022611,1.087847,0.825980,0.881072



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.02it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.12it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.78it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.73it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]


TrainOutput(global_step=690, training_loss=0.11195339423239879, metrics={'train_runtime': 134.706, 'train_samples_per_second': 81.689, 'train_steps_per_second': 5.122, 'total_flos': 444815961302640.0, 'train_loss': 0.11195339423239879, 'epoch': 3.0})

In [63]:
import torch

sentence1 = "The company released a new product."
sentence2 = "The firm launched a new product."

inputs = tokenizer(
    sentence1,
    sentence2,
    return_tensors="pt",
    truncation=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()

with torch.inference_mode():
    outputs = model(**inputs)

prediction = outputs.logits.argmax(dim=-1).item()

print(prediction)

1


# Practical Examples
Let’s work through some practical examples of learning curves. First, we will highlight some approaches to monitor the learning curves during training. Below, we will break down the different patterns that can be observed in the learning curves.

### During Training
During the training process (after you’ve hit trainer.train()), you can monitor these key indicators:

1. Loss convergence: Is the loss still decreasing or has it plateaued?
2. Overfitting signs: Is validation loss starting to increase while training loss decreases?
3. Learning rate: Are the curves too erratic (LR too high) or too flat (LR too low)?
4. Stability: Are there sudden spikes or drops that indicate problems?


### After Training
After the training process is complete, you can analyze the complete curves to understand the model’s performance.

1. Final performance: Did the model reach acceptable performance levels?
2. Efficiency: Could the same performance be achieved with fewer epochs?
3. Generalization: How close are training and validation performance?
4. Trends: Would additional training likely improve performance?

In [ ]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    num_train_epochs=10,  # Set high, but we'll stop early
)

# Add early stopping to prevent overfitting
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)